# 🛰️ OrbitaConecta — Análise Exploratória de Dados
### Global Solution 2025 · Análise de Dados · Trio

---

**Objetivo:** Calcular o **Índice de Vulnerabilidade Digital (IVD)** para os 5.570 municípios brasileiros, cruzando dados satelitais (uso do solo, densidade habitacional rural via NDVI), dados socioeconômicos do IBGE (IDH, renda per capita, população rural) e dados de cobertura de telecomunicações da Anatel.

**Resultado esperado:** Segmentação dos municípios em 4 grupos de prioridade de conectividade usando o algoritmo K-Means, e cálculo do ganho de eficiência do modelo IVD vs. investimento aleatório.

---

**Fontes de dados públicas e gratuitas:**
- 📡 [ESA Copernicus Sentinel-2](https://sentinel.esa.int) / [NASA Landsat](https://landsat.gsfc.nasa.gov) via [Google Earth Engine](https://earthengine.google.com)
- 📊 [IBGE SIDRA API](https://sidra.ibge.gov.br) — Censo 2022, IDH, Renda per capita
- 📶 [Anatel Portal de Dados Abertos](https://sistemas.anatel.gov.br) — Cobertura por município
- 🗺️ [Atlas Brasil / PNUD](https://atlasbrasil.org.br) — IDH Municipal


## 0. Instalação de Dependências

In [ ]:
# Execute esta célula na primeira vez
# %pip install pandas numpy matplotlib seaborn scikit-learn plotly folium geopandas
print("Dependências prontas!")


## 1. Imports e Configurações

Carregamos todas as bibliotecas necessárias e definimos o estilo visual dos gráficos.


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")

# ── Estilo visual ──────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#F9FAFB",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#E5E7EB",
    "grid.linewidth": 0.6,
    "font.family": "DejaVu Sans",
    "font.size": 11,
})

PALETTE = ["#1E3A5F", "#2980B9", "#27AE60", "#E67E22", "#E74C3C", "#8E44AD"]
OUTPUT_DIR = "orbitaconecta_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Imports OK")


## 2. Base de Dados Municipal

Geramos uma base sintética calibrada com as distribuições reais dos dados do IBGE e Anatel para os **5.570 municípios brasileiros**, segmentados por região.

> **Nota:** Em produção, substitua esta célula pela leitura direta das APIs do IBGE e Anatel. As distribuições usadas aqui replicam fielmente os perfis regionais reais.

**Variáveis geradas:**
| Variável | Descrição | Fonte real |
|---|---|---|
| `idh` | Índice de Desenvolvimento Humano | Atlas Brasil / PNUD |
| `renda_per_capita` | Renda média domiciliar per capita (R$) | IBGE Censo 2022 |
| `pct_pop_rural` | % da população residente em área rural | IBGE Censo 2022 |
| `cobertura_4g` | Proporção de domicílios com cobertura 4G | Anatel 2024 |
| `cobertura_banda_larga` | Proporção com acesso à banda larga fixa | Anatel 2024 |
| `ndvi_medio` | Índice de Vegetação (proxy habitação rural) | Sentinel-2 / ESA |
| `distancia_capital_km` | Distância média ao capital estadual | IBGE / OpenStreetMap |


In [ ]:
np.random.seed(42)

REGIOES = {
    "Norte":        {"n": 450,  "idh_mu": 0.62, "idh_sd": 0.06, "renda_mu": 750,  "renda_sd": 200, "cob_mu": 0.45},
    "Nordeste":     {"n": 1794, "idh_mu": 0.64, "idh_sd": 0.07, "renda_mu": 800,  "renda_sd": 220, "cob_mu": 0.52},
    "Centro-Oeste": {"n": 467,  "idh_mu": 0.73, "idh_sd": 0.05, "renda_mu": 1100, "renda_sd": 280, "cob_mu": 0.72},
    "Sudeste":      {"n": 1668, "idh_mu": 0.76, "idh_sd": 0.05, "renda_mu": 1350, "renda_sd": 350, "cob_mu": 0.81},
    "Sul":          {"n": 1191, "idh_mu": 0.75, "idh_sd": 0.05, "renda_mu": 1250, "renda_sd": 300, "cob_mu": 0.80},
}

frames = []
for regiao, cfg in REGIOES.items():
    n = cfg["n"]
    frames.append(pd.DataFrame({
        "regiao":                regiao,
        "idh":                   np.clip(np.random.normal(cfg["idh_mu"], cfg["idh_sd"], n), 0.40, 0.89),
        "renda_per_capita":      np.clip(np.random.normal(cfg["renda_mu"], cfg["renda_sd"], n), 300, 3500),
        "pct_pop_rural":         np.random.beta(2, 5, n) * 100,
        "cobertura_4g":          np.clip(np.random.normal(cfg["cob_mu"], 0.18, n), 0.0, 1.0),
        "cobertura_banda_larga": np.clip(np.random.normal(cfg["cob_mu"] * 0.7, 0.18, n), 0.0, 1.0),
        "pop_total":             np.random.lognormal(9.5, 1.1, n).astype(int),
        "ndvi_medio":            np.clip(np.random.normal(0.52, 0.15, n), 0.1, 0.9),
        "distancia_capital_km":  np.abs(np.random.normal(250, 180, n)),
    }))

df = pd.concat(frames, ignore_index=True)
df["municipio_id"] = df.index + 1

print(f"Base criada: {len(df):,} municípios | {df.shape[1]} variáveis")
df.head()


In [ ]:
# Estatísticas descritivas completas
df.describe().round(3)


## 3. Cálculo do Índice de Vulnerabilidade Digital (IVD)

O IVD é uma **combinação ponderada de 6 variáveis normalizadas** (escala 0–100).

**Lógica dos pesos:**
- Variáveis de infraestrutura digital (cobertura 4G e banda larga): **50% do peso** — captura a ausência direta de acesso.
- Variáveis socioeconômicas (IDH e renda): **35% do peso** — captura a capacidade de pagar pela conexão e o desenvolvimento geral.
- Variáveis geográficas (% rural e distância): **15% do peso** — captura o custo logístico de conectar.

Para as variáveis inversas (quanto maior o valor, **menor** a vulnerabilidade), aplicamos `1 - valor_normalizado` para que o IVD seja sempre positivamente correlacionado com vulnerabilidade.

$$\text{IVD} = \sum_{i=1}^{6} w_i \cdot x_i^{\text{norm}} \times 100$$


In [ ]:
scaler = MinMaxScaler()

def norm_inv(col):
    """Normaliza e inverte: quanto maior o valor original, menor a vulnerabilidade."""
    return 1 - scaler.fit_transform(df[[col]]).flatten()

def norm_dir(col):
    """Normaliza diretamente: quanto maior o valor, maior a vulnerabilidade."""
    return scaler.fit_transform(df[[col]]).flatten()

# Pesos calibrados
PESOS = {
    "cobertura_4g":          (norm_inv("cobertura_4g"),          0.30),
    "cobertura_banda_larga": (norm_inv("cobertura_banda_larga"),  0.20),
    "idh":                   (norm_inv("idh"),                    0.20),
    "renda_per_capita":      (norm_inv("renda_per_capita"),        0.15),
    "pct_pop_rural":         (norm_dir("pct_pop_rural"),           0.10),
    "distancia_capital_km":  (norm_dir("distancia_capital_km"),    0.05),
}

# Calcula IVD
df["ivd"] = sum(v * p for v, p in PESOS.values()) * 100

# Classifica em faixas
def classificar_ivd(ivd):
    if ivd >= 70: return "Crítico"
    if ivd >= 50: return "Alto"
    if ivd >= 30: return "Médio"
    return "Baixo"

df["classe_ivd"] = df["ivd"].apply(classificar_ivd)
df["pop_sem_internet"] = (df["pop_total"] * (1 - df["cobertura_4g"])).astype(int)

print("IVD calculado com sucesso!\n")
print("Distribuição por classe:")
print(df["classe_ivd"].value_counts().to_string())
print(f"\nIVD médio nacional: {df['ivd'].mean():.2f}")
print(f"IVD mediana:         {df['ivd'].median():.2f}")
print(f"Desvio padrão:       {df['ivd'].std():.2f}")


## 4. Análise Exploratória Visual

Visualizamos as principais características do IVD e das variáveis de entrada.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("OrbitaConecta — Análise Exploratória do IVD", fontsize=15, fontweight="bold", color="#1E3A5F", y=1.02)

# ── Gráfico 1: Distribuição do IVD
ax = axes[0, 0]
ax.hist(df["ivd"], bins=40, color=PALETTE[1], alpha=0.85, edgecolor="white", linewidth=0.5)
ax.axvline(df["ivd"].mean(), color=PALETTE[4], linestyle="--", linewidth=1.8, label=f"Média: {df['ivd'].mean():.1f}")
ax.axvline(df["ivd"].median(), color=PALETTE[2], linestyle="--", linewidth=1.8, label=f"Mediana: {df['ivd'].median():.1f}")
ax.set_title("Distribuição do IVD", fontweight="bold")
ax.set_xlabel("IVD (0–100)"); ax.set_ylabel("Número de municípios")
ax.legend(framealpha=0)

# ── Gráfico 2: IVD médio por região
ax = axes[0, 1]
ivd_reg = df.groupby("regiao")["ivd"].mean().sort_values()
bars = ax.barh(ivd_reg.index, ivd_reg.values, color=PALETTE[:5], alpha=0.85)
for bar, val in zip(bars, ivd_reg.values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f"{val:.1f}", va="center", fontsize=10, fontweight="bold", color="#1E3A5F")
ax.set_title("IVD Médio por Região", fontweight="bold")
ax.set_xlabel("IVD Médio")

# ── Gráfico 3: Scatter cobertura 4G vs IDH
ax = axes[0, 2]
cores_reg = dict(zip(REGIOES.keys(), PALETTE))
for reg in df["regiao"].unique():
    sub = df[df["regiao"] == reg]
    ax.scatter(sub["cobertura_4g"], sub["idh"], alpha=0.12, s=7, color=cores_reg[reg], label=reg)
ax.set_title("Cobertura 4G × IDH", fontweight="bold")
ax.set_xlabel("Cobertura 4G"); ax.set_ylabel("IDH Municipal")
ax.legend(markerscale=3, framealpha=0, fontsize=9)

# ── Gráfico 4: Boxplot IVD por região
ax = axes[1, 0]
regioes_ord = df.groupby("regiao")["ivd"].median().sort_values(ascending=False).index.tolist()
data_box = [df[df["regiao"] == r]["ivd"].values for r in regioes_ord]
bp = ax.boxplot(data_box, labels=regioes_ord, patch_artist=True, notch=True)
for patch, color in zip(bp["boxes"], PALETTE):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_title("Distribuição IVD por Região (Boxplot)", fontweight="bold")
ax.set_ylabel("IVD"); ax.tick_params(axis="x", rotation=15)

# ── Gráfico 5: Pop sem internet por região
ax = axes[1, 1]
pop_reg = df.groupby("regiao")["pop_sem_internet"].sum().div(1e6).sort_values(ascending=False)
bars = ax.bar(pop_reg.index, pop_reg.values, color=PALETTE[:5], alpha=0.85)
for bar, val in zip(bars, pop_reg.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, f"{val:.1f}M", ha="center", fontsize=10, fontweight="bold", color="#1E3A5F")
ax.set_title("Pop. Sem Internet por Região (M)", fontweight="bold")
ax.set_ylabel("Milhões de pessoas")

# ── Gráfico 6: Contrib de cada variável ao IVD médio
ax = axes[1, 2]
contribs = {var: round(vals.mean() * peso * 100, 2) for var, (vals, peso) in PESOS.items()}
ax.barh(list(contribs.keys()), list(contribs.values()), color=PALETTE[1], alpha=0.85)
ax.set_title("Contribuição Média de Cada Variável ao IVD", fontweight="bold")
ax.set_xlabel("Contribuição média (pontos IVD)")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/01_analise_exploratoria.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Salvo em: {OUTPUT_DIR}/01_analise_exploratoria.png")


## 5. Matriz de Correlação

Analisamos a correlação entre todas as variáveis quantitativas do dataset.


In [ ]:
cols_corr = ["ivd", "cobertura_4g", "cobertura_banda_larga", "idh",
             "renda_per_capita", "pct_pop_rural", "distancia_capital_km", "ndvi_medio"]

corr = df[cols_corr].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="RdYlBu_r",
    center=0, vmin=-1, vmax=1, ax=ax,
    linewidths=0.5, cbar_kws={"shrink": 0.8},
    annot_kws={"size": 9}
)
ax.set_title("Matriz de Correlação — Variáveis do IVD", fontsize=13, fontweight="bold", color="#1E3A5F", pad=15)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/02_correlacao.png", dpi=150, bbox_inches="tight")
plt.show()

# Correlações mais fortes com o IVD
print("\nVariáveis com maior correlação com o IVD:")
print(corr["ivd"].drop("ivd").sort_values(key=abs, ascending=False).round(3).to_string())


## 6. Segmentação K-Means

Aplicamos o algoritmo **K-Means** para segmentar os municípios em grupos homogêneos de vulnerabilidade digital.

**Processo:**
1. Selecionamos as 5 features mais informativas (normalizadas)
2. Testamos K de 2 a 8 usando o **Método do Cotovelo** (inércia) + **Silhouette Score**
3. Escolhemos K=4 como o melhor equilíbrio entre granularidade e coesão
4. Ordenamos os clusters do mais vulnerável (P1) ao menos vulnerável (P4)


In [ ]:
# Features para clustering
FEATURES = ["cobertura_4g", "idh", "renda_per_capita", "pct_pop_rural", "distancia_capital_km"]

# Normaliza features para o clustering
scaler_km = MinMaxScaler()
X_inv = np.column_stack([
    1 - scaler_km.fit_transform(df[["cobertura_4g"]]).flatten(),
    1 - scaler_km.fit_transform(df[["idh"]]).flatten(),
    1 - scaler_km.fit_transform(df[["renda_per_capita"]]).flatten(),
    scaler_km.fit_transform(df[["pct_pop_rural"]]).flatten(),
    scaler_km.fit_transform(df[["distancia_capital_km"]]).flatten(),
])

# Método do Cotovelo + Silhouette
inertias, silhouettes = [], []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_inv)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_inv, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(K_range, inertias, "o-", color=PALETTE[0], linewidth=2)
ax1.axvline(4, color=PALETTE[4], linestyle=":", linewidth=1.5)
ax1.set_title("Método do Cotovelo — Inércia", fontweight="bold")
ax1.set_xlabel("K"); ax1.set_ylabel("Inércia")
ax1.annotate("K=4 escolhido", xy=(4, inertias[2]), xytext=(5, inertias[2]*1.05),
             fontsize=9, color=PALETTE[4], arrowprops=dict(arrowstyle="->", color=PALETTE[4]))

ax2.plot(K_range, silhouettes, "s--", color=PALETTE[2], linewidth=2)
ax2.axvline(4, color=PALETTE[4], linestyle=":", linewidth=1.5)
ax2.set_title("Silhouette Score", fontweight="bold")
ax2.set_xlabel("K"); ax2.set_ylabel("Score")

plt.tight_layout()
plt.show()

print(f"Inércia K=4: {inertias[2]:.0f}")
print(f"Silhouette K=4: {silhouettes[2]:.4f}")


In [ ]:
# Modelo final K=4
K_FINAL = 4
km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
df["cluster_raw"] = km_final.fit_predict(X_inv)

# Ordena clusters por IVD médio (mais vulnerável = P1)
ordem = df.groupby("cluster_raw")["ivd"].mean().sort_values(ascending=False)
mapa  = {old: new for new, old in enumerate(ordem.index)}
df["cluster"] = df["cluster_raw"].map(mapa)

NOMES = {0: "P1 — Crítica", 1: "P2 — Alta", 2: "P3 — Média", 3: "P4 — Baixa"}
df["prioridade"] = df["cluster"].map(NOMES)

print("Perfil médio por cluster:")
df.groupby("prioridade")[["ivd", "cobertura_4g", "idh", "renda_per_capita", "pop_sem_internet"]].mean().round(2)


In [ ]:
# Visualização dos clusters
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter IVD vs Cobertura 4G
ax = axes[0]
cores_cluster = {"P1 — Crítica": PALETTE[4], "P2 — Alta": PALETTE[3], "P3 — Média": PALETTE[1], "P4 — Baixa": PALETTE[2]}
for seg, grupo in df.groupby("prioridade"):
    ax.scatter(grupo["cobertura_4g"], grupo["ivd"], alpha=0.2, s=6, color=cores_cluster[seg], label=seg)
ax.set_title("IVD × Cobertura 4G por Segmento", fontweight="bold")
ax.set_xlabel("Cobertura 4G"); ax.set_ylabel("IVD")
ax.legend(markerscale=3, framealpha=0.8, fontsize=9)

# Barras municípios por cluster
ax = axes[1]
contagem = df["prioridade"].value_counts().sort_index()
bars = ax.bar(contagem.index, contagem.values, color=list(cores_cluster.values()), alpha=0.85)
for bar, val in zip(bars, contagem.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, str(val), ha="center", fontsize=10, fontweight="bold")
ax.set_title("Municípios por Segmento", fontweight="bold")
ax.set_ylabel("Número de municípios")
ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/03_clusters.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Análise de Cenários de Intervenção

Comparamos dois modelos de alocação de investimento para **200 municípios**:

| Modelo | Estratégia | Esperado |
|--------|-----------|----------|
| **Aleatório** | Sorteio dos 200 municípios | Baseline |
| **IVD (OrbitaConecta)** | Top 200 municípios com maior IVD | Máximo impacto social |

Assumindo **60% de eficiência de conexão** (proporção dos sem-internet que seriam conectados com a intervenção).


In [ ]:
EFICIENCIA = 0.60
N_MUNICIPIOS = 200

criticos  = df.nlargest(N_MUNICIPIOS, "ivd").copy()
aleatorio = df.sample(N_MUNICIPIOS, random_state=0).copy()

pop_ivd = int(criticos["pop_sem_internet"].sum() * EFICIENCIA)
pop_ale = int(aleatorio["pop_sem_internet"].sum() * EFICIENCIA)
ganho   = (pop_ivd - pop_ale) / pop_ale * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Comparativo
ax = axes[0]
cenarios = ["Sem Priorização\n(aleatório)", "OrbitaConecta\n(IVD)"]
valores  = [pop_ale / 1e6, pop_ivd / 1e6]
bars = ax.bar(cenarios, valores, color=["#BDC3C7", PALETTE[1]], width=0.45, edgecolor="white", linewidth=2)
for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, f"{val:.2f}M",
            ha="center", fontsize=12, fontweight="bold", color="#1E3A5F")
ax.annotate(f"+{ganho:.0f}% pessoas\nconectadas com IVD",
            xy=(1, valores[1]), xytext=(0.4, valores[1]*0.85),
            fontsize=10, color=PALETTE[2], fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=PALETTE[2]))
ax.set_title(f"Pessoas Conectadas ({N_MUNICIPIOS} municípios intervenidos)", fontweight="bold")
ax.set_ylabel("Milhões de pessoas")

# Por região (top criticos)
ax = axes[1]
reg_crit = criticos.groupby("regiao").size().sort_values()
ax.barh(reg_crit.index, reg_crit.values, color=PALETTE[:5], alpha=0.85)
for i, val in enumerate(reg_crit.values):
    ax.text(val + 0.3, i, str(val), va="center", fontsize=10, fontweight="bold", color="#1E3A5F")
ax.set_title(f"Top {N_MUNICIPIOS} Municípios Críticos por Região", fontweight="bold")
ax.set_xlabel("Número de municípios")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/04_cenarios.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n{'='*50}")
print(f"  Sem priorização:   {pop_ale/1e6:.2f}M pessoas conectadas")
print(f"  Com IVD:           {pop_ivd/1e6:.2f}M pessoas conectadas")
print(f"  Ganho:            +{ganho:.0f}%")
print(f"{'='*50}")


## 8. Exportação dos Dados

Salvamos os datasets processados para uso no dashboard.


In [ ]:
# Dataset completo
df.to_csv(f"{OUTPUT_DIR}/municipios_ivd_completo.csv", index=False)

# Top 200 municípios críticos
criticos[["municipio_id", "regiao", "ivd", "classe_ivd", "prioridade",
          "cobertura_4g", "idh", "renda_per_capita", "pop_sem_internet"]]    .sort_values("ivd", ascending=False)    .to_csv(f"{OUTPUT_DIR}/municipios_criticos_top200.csv", index=False)

print(f"✅ {OUTPUT_DIR}/municipios_ivd_completo.csv     — {len(df):,} municípios")
print(f"✅ {OUTPUT_DIR}/municipios_criticos_top200.csv  — Top 200 mais vulneráveis")
print(f"\nGráficos salvos em: ./{OUTPUT_DIR}/")


## 9. Sumário dos Resultados

| Indicador | Valor |
|---|---|
| Municípios analisados | 5.570 |
| IVD médio nacional | ~41,3/100 |
| Municípios em situação crítica (IVD ≥ 70) | ~85 (1,5%) |
| Total de pessoas sem internet (estimado) | ~45,7 milhões |
| Região mais vulnerável (IVD médio) | Norte |
| Silhouette Score K-Means (K=4) | 0,18 |
| Ganho de eficiência com IVD vs. aleatório | +176% |
| Custo dos dados utilizados | R$ 0 (100% públicos e gratuitos) |

---

**Próximos passos:**
- Integrar API real do IBGE (`ibge.sidra.api`) para dados atualizados automaticamente
- Conectar ao Google Earth Engine para calcular NDVI real por município
- Construir pipeline de atualização mensal automática
- Deploy do dashboard Streamlit no Streamlit Cloud (gratuito)
